# <a id='toc1_'></a>[Multilayer Perceptron for Iris Classification](#toc0_)

A Multilayer Perceptron (MLP) is a feed-forward neural network made of fully connected layers. In this notebook, the MLP learns to classify an Iris flower into one of three classes from four numerical features: sepal length, sepal width, petal length, and petal width.

The network maps an input vector $x \in \mathbb{R}^4$ to three output logits. The class with the largest logit is selected as the prediction.

## <a id='toc1_1_'></a>[Import the Required Libraries](#toc0_)

NumPy and PyTorch are used for numerical computation and model training. Scikit-learn provides the Iris dataset, the train-test split, feature standardization, and the accuracy metric.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

## <a id='toc1_2_'></a>[Load and Prepare the Iris Dataset](#toc0_)

The Iris dataset contains four input features and three target classes. The data is divided into training and test sets using an 80/20 split. `stratify=y` keeps the class proportions similar in both sets.

Each feature is standardized using statistics computed from the training set only:

$$
x' = \frac{x - \mu_{train}}{\sigma_{train}}
$$

The same training mean and standard deviation are then applied to the test set. This prevents information from the test set from leaking into preprocessing. Finally, the NumPy arrays are converted to PyTorch tensors.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

## <a id='toc1_3_'></a>[Define, Train, and Evaluate the MLP](#toc0_)

The network has the following architecture:

$$
4 \rightarrow 16 \rightarrow 8 \rightarrow 3
$$

The first two linear layers are followed by ReLU activation:

$$
h^{(1)} = \operatorname{ReLU}(xW_1 + b_1),
$$

$$
h^{(2)} = \operatorname{ReLU}(h^{(1)}W_2 + b_2),
$$

where $\operatorname{ReLU}(z) = \max(0,z)$. The final layer produces three logits:

$$
z = h^{(2)}W_3 + b_3.
$$

The layer parameter shapes are $W_1 \in \mathbb{R}^{4 \times 16}$, $W_2 \in \mathbb{R}^{16 \times 8}$, and $W_3 \in \mathbb{R}^{8 \times 3}$. Biases have shapes $[16]$, $[8]$, and $[3]$.

`CrossEntropyLoss` receives the raw logits and internally applies the log-softmax operation. Therefore, a separate `Softmax` layer is not needed during training. For one example with target class $y$, the loss is:

$$
\mathcal{L} = -\log \left( \frac{e^{z_y}}{\sum_{k=1}^{3} e^{z_k}} \right).
$$

For each epoch, the optimizer clears old gradients, computes the forward pass, evaluates the loss, runs backpropagation, and updates the parameters with Adam:

$$
\theta \leftarrow \operatorname{AdamUpdate}(\theta, \nabla_{\theta}\mathcal{L}).
$$

During evaluation, gradients are disabled, the class with the largest logit is selected with `argmax`, and accuracy is computed on the test set.

**Table of contents**<a id='toc0_'></a>    
- [Multilayer Perceptron for Iris Classification](#toc1_)    
  - [Import the Required Libraries](#toc1_1_)    
  - [Load and Prepare the Iris Dataset](#toc1_2_)    
  - [Define, Train, and Evaluate the MLP](#toc1_3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(4, 16), # 16 layer, each layer has 4 weights -> output 16 values
            nn.ReLU(),
            nn.Dropout(p=0.5),

            nn.Linear(16, 8), # 8 layer, each Layer has 16 weights -> output 8 values
            nn.ReLU(),
            nn.Dropout(p=0.5),

            nn.Linear(8, 3) # 3 layer, each layer has 8 weights -> output 3 values
            # nn.Softmax(dim = 1)
        )
        # total weights = (14 * 4 + 16) + (17 * 8 + 8) + (8 * 3 + 3)

    def forward(self, x):
        return self.model(x)

model = MLP()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


epochs = 100
for epoch in range(epochs):
    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epoch}, Loss: {loss.item()}")
    if loss < 1e-2:
        break

model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    y_pred_classes = torch.argmax(y_pred, dim=1)
    accuracy = accuracy_score(y_test, y_pred_classes)

print(f"ACcuracy on test set: {accuracy:.4f}")

Epoch 10/9, Loss: 0.9030236005783081
Epoch 20/19, Loss: 0.5109954476356506
Epoch 30/29, Loss: 0.3346462845802307
Epoch 40/39, Loss: 0.23602858185768127
Epoch 50/49, Loss: 0.13582158088684082
Epoch 60/59, Loss: 0.07304205745458603
Epoch 70/69, Loss: 0.04868567734956741
Epoch 80/79, Loss: 0.03956284746527672
Epoch 90/89, Loss: 0.03530442342162132
Epoch 100/99, Loss: 0.03249790146946907
ACcuracy on test set: 0.9667
